# Chapter 7: fill weights and giveaway

Reads one week of synthetic checkweigher readings from the can line (three shifts a day, one can from each of 40 filler heads), plots the mean and spread by shift and prices the giveaway.

The sum: giveaway in hL = packs x mean overfill in ml / 100,000.

Synthetic Kestrel Brewing data. No real brewery's numbers. Book rates: USD 1 = INR 90; EUR 1 = USD 1.10.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

f = pd.read_csv("../data/fill_weights.csv", parse_dates=["date"])
DECLARED_ML = 330
by_shift = f.groupby(["date", "shift"]).fill_ml.agg(["mean", "std", "min"]).round(2)
by_shift.head(6)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.5))
x = range(len(by_shift))
ax.errorbar(x, by_shift["mean"], yerr=by_shift["std"], fmt="o", color="#0b4f45", ecolor="#2f7f72", capsize=3)
ax.axhline(DECLARED_ML, color="grey", ls="--")
ax.set_xticks(list(x), [f"{d:%a} {s}" for d, s in by_shift.index], rotation=90)
ax.set_ylabel("Fill, ml (mean and one standard deviation)")
ax.set_title("Fill by shift, can line, one week (synthetic)")
plt.tight_layout()
plt.show()

## Fix the head before the target

One filler head running high shows up here before it shows in the shift means.

In [ ]:
heads = f.groupby("filler_head").fill_ml.mean().sub(f.fill_ml.mean()).sort_values(ascending=False)
heads.head(5).round(2)

## The week in money (chapters 7.2 and 7.5)

In [ ]:
daily = f.groupby("date").fill_ml.mean()
print("Daily means:", ", ".join(f"{v:.1f}" for v in daily))
overfill = round(f.fill_ml.mean() - DECLARED_ML, 1)
cans_a_year_m = round(35000 / 0.33 * 100 / 1e6, 1)
giveaway_hl = round(cans_a_year_m * 1e6 * overfill / 100000)
per_ml_hl = cans_a_year_m * 1e6 / 100000
print(f"Mean overfill {overfill:.1f} ml on {cans_a_year_m} million cans a year: {giveaway_hl} hL ({100 * overfill / DECLARED_ML:.1f} percent of volume)")
print(f"Each ml is {per_ml_hl:.0f} hL a year")
usd = giveaway_hl * 32
print(f"Replacement cost USD {usd:,.0f} (INR {usd * 90:,.0f}; EUR {usd / 1.1:,.0f}); per ml USD {usd / overfill:,.0f}")
print(f"Upper bound USD {giveaway_hl * 112:,.0f}; at price USD {giveaway_hl * 150:,.0f} (never use price)")
kept = giveaway_hl / 2
print(f"Halve the overfill: keep {kept:.0f} hL, USD {kept * 32:,.0f} a year (upper bound USD {kept * 112:,.0f})")

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3))
ax.plot(daily.index.strftime("%a"), daily, "o-", color="#0b4f45")
ax.fill_between(daily.index.strftime("%a"), daily, DECLARED_ML, color="#c27c0e", alpha=0.35)
ax.axhline(DECLARED_ML, color="grey", ls="--")
ax.set_ylim(328, 338)
ax.set_ylabel("Mean fill (ml per can)")
ax.set_title("Figure 7.2: giveaway against the declared 330 ml (synthetic)")
plt.tight_layout()
plt.show()